# Multi-Agent AI Music Producer - Colab Test (TPU)

This notebook runs the full workflow with HuggingFace models on **TPU v5e**.

**Requirements:** TPU runtime
- Runtime → Change runtime type → TPU v5e

In [ ]:
# 1. Check TPU availability
import os

# Check if TPU is available
if 'COLAB_TPU_ADDR' in os.environ:
    print(f"✅ TPU available at: {os.environ['COLAB_TPU_ADDR']}")
else:
    # For newer Colab TPU v5e
    try:
        import torch_xla
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
        print(f"✅ TPU device: {device}")
    except:
        print("⚠️ TPU not detected. Make sure you selected TPU runtime.")

In [ ]:
# 2. Mount Google Drive and clone repo (persists between sessions!)
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive

# Clone repo (only first time - comment out after)
!git clone https://github.com/YOUR_USERNAME/multi_agent_ai_music_producer.git

# If already cloned, just pull latest:
# %cd /content/drive/MyDrive/multi_agent_ai_music_producer && git pull

In [ ]:
# 3. Go to project directory
%cd /content/drive/MyDrive/multi_agent_ai_music_producer

In [ ]:
# 4. Install dependencies for TPU

# Upgrade pip first
!pip install --upgrade pip setuptools wheel -q

# Install PyTorch XLA for TPU (Colab TPU v5e should have this)
# If not already installed:
# !pip install torch torch_xla -q

# Install core dependencies
!pip install pydantic pydantic-settings pyyaml python-dotenv -q
!pip install numpy scipy soundfile -q
!pip install transformers accelerate -q  # No bitsandbytes on TPU
!pip install nest_asyncio -q

# Install LangGraph
!pip install langgraph langchain-core -q

# Install the project
!pip install -e . --no-deps -q

# Add project to path
import sys
sys.path.insert(0, '/content/drive/MyDrive/multi_agent_ai_music_producer')

print("✅ Dependencies installed!")
print("If you see import errors, click Runtime → Restart runtime, then run from cell 5")

In [ ]:
# 5. Set HuggingFace token and verify imports
import os
import sys

# Add project to path
sys.path.insert(0, '/content/drive/MyDrive/multi_agent_ai_music_producer')

# Set your HuggingFace token
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # Replace with your token

# Check torch device availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Try TPU
try:
    import torch_xla.core.xla_model as xm
    tpu_device = xm.xla_device()
    print(f"✅ TPU device: {tpu_device}")
    USE_TPU = True
except ImportError:
    print("⚠️ torch_xla not available, will use CPU/GPU")
    USE_TPU = False

# Verify all imports work
try:
    from src.config import Settings, LLMConfig
    from src.llm.base import LLMMessage
    from src.llm.huggingface_provider import HuggingFaceProvider
    from src.logging.logger import MusicProducerLogger, LogLevel
    from src.graph.workflow import MusicProducerGraph
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Try: Runtime → Restart runtime, then run cells 4-5 again")

In [ ]:
# 6. Test HuggingFace provider on TPU (loads model - takes ~2 min)
import asyncio

# Enable async in Colab
import nest_asyncio
nest_asyncio.apply()

# For TPU: Use bfloat16 (native TPU dtype), no quantization
provider = HuggingFaceProvider(
    model="Qwen/Qwen2.5-7B-Instruct",
    token=os.environ["HF_TOKEN"],
    # TPU settings
    load_in_4bit=False,  # No quantization on TPU
    load_in_8bit=False,
    torch_dtype="bfloat16",  # Best for TPU
    device="auto",
    temperature=0.7,
    max_tokens=512,
)

# Test simple generation
messages = [LLMMessage(role="user", content="Say hello in exactly 3 words")]

async def test_gen():
    return await provider.generate(messages)

response = asyncio.get_event_loop().run_until_complete(test_gen())
print(f"✅ Model loaded and working!")
print(f"Response: {response.content}")

In [ ]:
# 7. Setup workflow
from pathlib import Path
from unittest.mock import patch
import numpy as np
import soundfile as sf

from src.config import Settings, LLMConfig, GenerationConfig, AudioConfig, LoggingConfig
from src.logging.logger import MusicProducerLogger, LogLevel
from src.graph.workflow import MusicProducerGraph

output_dir = Path("/content/output")
output_dir.mkdir(exist_ok=True)

# Mock audio generator
class MockAudioGenerator:
    def __init__(self, sample_rate=32000):
        self.sample_rate = sample_rate
        self.call_count = 0
    
    def generate(self, prompt, duration_sec, output_path, **kwargs):
        self.call_count += 1
        t = np.linspace(0, duration_sec, int(self.sample_rate * duration_sec))
        audio = 0.3 * np.sin(2 * np.pi * (220 + self.call_count * 55) * t)
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        sf.write(output_path, audio, self.sample_rate)
        return {"success": True, "path": output_path}

mock_audio = MockAudioGenerator()

settings = Settings(
    llm=LLMConfig(
        provider="huggingface",
        model="Qwen/Qwen2.5-7B-Instruct",
        temperature=0.7,
        max_tokens=1024,
    ),
    generation=GenerationConfig(max_retries=1, default_segment_duration=5.0),
    audio=AudioConfig(sample_rate=32000),
    logging=LoggingConfig(level="DEBUG", output_dir=str(output_dir / "logs")),
)

logger = MusicProducerLogger(
    run_id="colab_run",
    output_dir=str(output_dir),
    level=LogLevel.DEBUG,
    console_output=True,
)

print(f"Ready! Model: {settings.llm.model}")

In [ ]:
# 8. Run the workflow!
with patch("src.tools.audio_generation.generate_segment") as mock_gen:
    mock_gen.side_effect = lambda prompt, duration_sec, output_path, **kw: \
        mock_audio.generate(prompt, duration_sec, output_path, **kw)
    
    graph = MusicProducerGraph(settings=settings, logger=logger)
    graph.build()
    
    print("\n" + "="*50)
    print("Starting Music Production")
    print("="*50 + "\n")
    
    result = graph.invoke(
        user_prompt="Create a chill lofi hip-hop beat with jazzy piano and mellow drums",
        reference_paths=[],
    )
    
    print("\n" + "="*50)
    print("Done!")
    print(f"Phase: {result.get('phase')}")
    print(f"Plan: {result.get('track_plan')}")

In [ ]:
# 9. Play the audio (if generated)
from IPython.display import Audio
if result.get('final_track_path'):
    Audio(result['final_track_path'])